In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gold_staging_df = (
    validated_df
        .withColumn(
            "revenue",
            col("quantity").cast("int") *
            col("unit_price").cast("double")
        )
)

sales_summary_df = (
    gold_staging_df.agg(
        count("*").alias("total_orders"),
        approx_count_distinct("customer_id").alias("total_customers"),
        sum("revenue").alias("total_revenue"),
        sum(
            when(
                col("order_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias("delivered_orders")
    )
)

gold_table = "retail_lakehouse.gold.sales_summary"

gold_stream = (
    sales_summary_df.writeStream
        .format("delta")
        .outputMode("complete")
        .option(
            "checkpointLocation",
            "/Volumes/retail_lakehouse/bronze/raw_files/checkpoints/sales_summary_gold/"
        )
        .trigger(availableNow=True)
        .toTable(gold_table)
)